# SEACO-6161 · 04 Cost report · `dbw-opas-merone-pipeline-prod`

**Read-only.** Answers *"what is costing us on `dbstorageld3xv3o6mfr6o` (188.41 TiB, $2,189/month)?"* from the audit tables
that `01_investigation` and `01a_dbfs_scan` already saved. It lists nothing and sizes nothing again, so it
takes a minute or two on any cluster that can run `%run ./00_config_and_helpers`.

Every size is turned into **$/month** at the account's effective rate (Azure monthly cost ÷ Azure size).

| Section | Question |
|---|---|
| 1 | Which audit runs is this report built on, and how complete are they? |
| 2 | Where does the money go, split into buckets that do not overlap? |
| 3 | Which `dbfs:/` directories cost most, and how much of each is not in any registered table? |
| 4 | Which databases cost most? |
| 5 | Which tables cost most, and what should happen to each (VACUUM / DROP / keep)? |
| 6 | Orphan directories, log destinations and the jobs that write the expensive tables |
| 7 | Savings summary, printed as Markdown to paste into the ticket |

⚠️ The report uses the **latest** run of each audit table. If the latest `01_investigation` run was limited
with the `databases` widget, the table figures cover only those databases. Section 1 shows this.

In [0]:
%run ./00_config_and_helpers

## 1 · Audit runs used and the effective rate

In [0]:
RATE = AZURE_MONTHLY_COST_USD / (AZURE_REPORTED_TIB * TIB)   # $ per byte-month
AZURE_BYTES = AZURE_REPORTED_TIB * TIB


def usd(b):
    return 0.0 if b is None else float(b) * RATE


def latest_or_none(name):
    try:
        return latest(name)
    except Exception as e:
        print(f"  ! {AUDIT_DB}.{name}: {str(e)[:150]}")
        return None, None


inv, inv_ts = latest_or_none("table_inventory")
scan, scan_ts = latest_or_none("dbfs_scan")
plan, _ = latest_or_none("dbfs_scan_plan")
logs, _ = latest_or_none("cluster_log_conf")
jobs, _ = latest_or_none("jobs_inventory")
orph, _ = latest_or_none("orphan_dirs")
if inv is None:
    raise RuntimeError("No table_inventory run found: run 01_investigation first")

# Normalised path of each table location (dbfs:/a/b or abfss://root@acct.../a/b  ->  /a/b)
inv = inv.withColumn("npath", F.regexp_replace(
    F.when(F.col("location").startswith("abfss://"), F.regexp_replace("location", r"^abfss://[^/]+", ""))
     .otherwise(F.regexp_replace("location", r"^dbfs:", "")), r"/+$", ""))
inv = inv.withColumn("stale", F.col("last_modified") < F.expr("current_timestamp() - INTERVAL 30 DAYS"))
inv = inv.withColumn("shallow_src", F.coalesce("is_shallow_clone_source", F.lit(False)))
inv.cache().createOrReplaceTempView("inv")

scan_run = None
if scan is not None:
    scan_run = scan.select("run_id").first()[0]
    scan = scan.dropDuplicates(["path"]).cache()
    scan.createOrReplaceTempView("scan")
    planned = plan.filter("kind = 'dir'").count() if plan is not None else 0
    plan_run = plan.select("run_id").first()[0] if plan is not None else None
    sized = scan.filter("kind = 'dir'").count()
    SCAN_COVERAGE = sized / max(planned, 1) if plan_run == scan_run else None
else:
    SCAN_COVERAGE = None

if logs is not None:
    logs.createOrReplaceTempView("log_conf")
if jobs is not None:
    jobs.createOrReplaceTempView("jobs")
if orph is not None:
    orph.createOrReplaceTempView("orphans")
spark.udf.register("usd", lambda b: None if b is None else round(float(b) * RATE, 2), "double")

In [0]:
inv_run = inv.select("run_id").first()[0]
n_db = inv.select("database").distinct().count()
print(f"Effective rate          : ${RATE * GIB:.5f} per GiB-month  (${AZURE_MONTHLY_COST_USD:,.2f} / {AZURE_REPORTED_TIB} TiB)")
print(f"table_inventory run    : {inv_run} @ {inv_ts}  ({inv.count():,} tables in {n_db} databases)")
if scan_run:
    cov = "unknown (plan and scan runs differ)" if SCAN_COVERAGE is None else f"{SCAN_COVERAGE:.1%}"
    print(f"dbfs_scan run          : {scan_run} @ {scan_ts}  coverage {cov}")
    if SCAN_COVERAGE is not None and SCAN_COVERAGE < 0.999:
        print(f"  ⚠️  scan incomplete: directory figures are a lower bound. Resume 01a with resume_run_id = {scan_run}")
else:
    print("dbfs_scan run          : none. Run 01a_dbfs_scan to get the directory breakdown")

## 2 · Where the money goes

The account split into buckets that **do not overlap**, so the dollar figures add up to the Azure cost:

- **A** stale tables are counted whole (dropping them frees live and old versions alike). **B** is only the
  overhead of tables that are still written to. **D** is kept apart because VACUUM on a shallow-clone source
  breaks its clones.
- **G** is what the `dbfs:/` scan sees outside every registered table. Section 3 breaks it down per directory.
- **H** is what a notebook cannot see at all: the `jobs`, `logs`, `ephemeral` and `meta` containers and
  soft-deleted blobs. Check it in the Azure portal (storage account → Containers / Metrics by container).

In [0]:
def one(sql):
    v = spark.sql(sql).collect()[0][0]
    return int(v or 0)


root = "storage_class = 'root'"
table_disk = one(f"SELECT sum(disk_bytes) FROM inv WHERE {root}")
table_live = one(f"SELECT sum(live_bytes) FROM inv WHERE {root}")
table_log = one(f"SELECT sum(delta_log_bytes) FROM inv WHERE {root}")
stale_disk = one(f"SELECT sum(disk_bytes) FROM inv WHERE {root} AND stale AND NOT shallow_src")
active_over = one(f"SELECT sum(overhead_bytes) FROM inv WHERE {root} AND NOT coalesce(stale, false) AND NOT shallow_src")
shallow_disk = one(f"SELECT sum(disk_bytes) FROM inv WHERE {root} AND shallow_src")
active_keep = table_disk - stale_disk - active_over - shallow_disk  # live + _delta_log of active tables
visible = one("SELECT sum(bytes) FROM scan WHERE storage_class = 'root'") if scan_run else None
orphan_b = one("SELECT sum(bytes) FROM orphans") if orph is not None else 0
log_b = one("SELECT sum(b) FROM (SELECT destination, max(destination_bytes) b FROM log_conf "
            "WHERE storage_class = 'root' GROUP BY destination)") if logs is not None else 0

BUCKET_ROWS = [
    ("A", "Stale tables (no write for 30+ days): DROP candidates", stale_disk, "drop after owner confirms"),
    ("B", "Active tables: old file versions VACUUM can remove", active_over, "VACUUM RETAIN 168 HOURS"),
    ("C", "Active tables: live data + _delta_log", active_keep, "keep"),
    ("D", "Shallow-clone sources (VACUUM would break clones)", shallow_disk, "fix the clones first"),
    ("E", "Orphan directories (no table owns them)", orphan_b, "delete after review"),
    ("F", "Cluster-log destinations on dbfs:/", log_b, "prune, set retention"),
]
if visible is not None:
    other = visible - table_disk - orphan_b - log_b
    BUCKET_ROWS.append(("G", "Other dbfs:/ data not in any table (tmp, FileStore, mnt dirs, …): see section 3",
                        other, "review per directory"))
    BUCKET_ROWS.append(("H", "Not visible from notebooks (jobs/logs/ephemeral/meta containers, soft-delete)",
                        AZURE_BYTES - visible, "Azure portal / Databricks support"))

df = spark.createDataFrame(
    [(k, n, round(b / TIB, 2), round(100 * b / AZURE_BYTES, 1), round(usd(b), 0), a) for k, n, b, a in BUCKET_ROWS],
    "bucket string, what string, tib double, pct_of_account double, usd_per_month double, action string")
display(df.orderBy(F.desc("usd_per_month")))

In [0]:
print(f"{'Account (Azure, May 2026)':<62} {AZURE_REPORTED_TIB:>9.2f} TiB  ${AZURE_MONTHLY_COST_USD:>9,.0f}/mo")
if visible is not None:
    print(f"{'Visible through dbfs:/':<62} {visible / TIB:>9.2f} TiB  ${usd(visible):>9,.0f}/mo")
print(f"{'Registered tables on disk':<62} {table_disk / TIB:>9.2f} TiB  ${usd(table_disk):>9,.0f}/mo")
print(f"{'  of which live data':<62} {table_live / TIB:>9.2f} TiB  ${usd(table_live):>9,.0f}/mo")
for k, n, b, a in sorted(BUCKET_ROWS, key=lambda r: -r[2]):
    print(f"  {k}  {n[:57]:<57} {b / TIB:>9.2f} TiB  ${usd(b):>9,.0f}/mo  -> {a}")
if visible is not None and visible < table_disk:
    print("\n⚠️  dbfs:/ scan total is below the table total: the scan is incomplete, bucket G is not reliable.")

## 3 · Most expensive `dbfs:/` directories

`not_in_tables_tib` is the part of a directory that no registered table owns. That is where bucket G lives.

In [0]:
# Directory rollup (levels 1-3) with the share held by registered tables under each prefix
if scan_run:
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW dir_cost AS
    WITH lv AS (SELECT explode(sequence(1, 3)) AS level),
    d AS (
      SELECT lv.level, concat_ws('/', slice(split(s.path, '/'), 1, lv.level + 1)) AS prefix,
             sum(s.bytes) AS bytes, sum(s.files) AS files
      FROM scan s CROSS JOIN lv
      WHERE s.storage_class = 'root' AND size(split(s.path, '/')) > lv.level
      GROUP BY ALL),
    t AS (
      SELECT lv.level, concat_ws('/', slice(split(i.npath, '/'), 1, lv.level + 1)) AS prefix,
             sum(i.disk_bytes) AS table_bytes, count(*) AS tables,
             sum(CASE WHEN i.stale THEN i.disk_bytes END) AS stale_table_bytes,
             sum(CASE WHEN NOT coalesce(i.stale, false) THEN i.overhead_bytes END) AS active_overhead_bytes
      FROM inv i CROSS JOIN lv
      WHERE i.storage_class = 'root' AND size(split(i.npath, '/')) > lv.level
      GROUP BY ALL)
    SELECT d.level, d.prefix,
           round(d.bytes / power(1024, 4), 3)                                   AS tib,
           usd(d.bytes)                                                         AS usd_per_month,
           d.files,
           coalesce(t.tables, 0)                                                AS tables,
           round(coalesce(t.table_bytes, 0) / power(1024, 4), 3)               AS in_tables_tib,
           round(greatest(d.bytes - coalesce(t.table_bytes, 0), 0) / power(1024, 4), 3) AS not_in_tables_tib,
           usd(greatest(d.bytes - coalesce(t.table_bytes, 0), 0))              AS not_in_tables_usd,
           round(coalesce(t.stale_table_bytes, 0) / power(1024, 4), 3)         AS stale_tables_tib,
           round(coalesce(t.active_overhead_bytes, 0) / power(1024, 4), 3)     AS vacuumable_tib
      FROM d LEFT JOIN t ON d.level = t.level AND d.prefix = t.prefix
    """)
    display(spark.sql("SELECT * FROM dir_cost WHERE level = 1 ORDER BY usd_per_month DESC"))
else:
    print("No dbfs_scan run: skipped")

In [0]:
%sql
-- Every directory (levels 2-3) costing $20/month or more. A big not_in_tables_tib means plain files:
-- tmp output, FileStore uploads, df.write.save() leftovers, mnt dirs that are not real mounts.
SELECT level, prefix, tib, usd_per_month, files, tables, in_tables_tib, not_in_tables_tib, not_in_tables_usd,
       stale_tables_tib, vacuumable_tib
FROM dir_cost
WHERE level >= 2 AND usd_per_month >= 20
ORDER BY usd_per_month DESC
LIMIT 100

## 4 · Most expensive databases

In [0]:
%sql
SELECT database,
       count(*)                                                           AS tables,
       round(sum(disk_bytes) / power(1024, 4), 3)                         AS disk_tib,
       usd(sum(disk_bytes))                                               AS usd_per_month,
       round(sum(live_bytes) / power(1024, 4), 3)                         AS live_tib,
       round(sum(disk_bytes) / nullif(sum(live_bytes), 0), 1)             AS disk_to_live,
       usd(sum(CASE WHEN stale AND NOT shallow_src THEN disk_bytes END))  AS stale_tables_usd,
       usd(sum(CASE WHEN NOT coalesce(stale, false) AND NOT shallow_src THEN overhead_bytes END)) AS vacuumable_usd,
       sum(CASE WHEN stale THEN 1 ELSE 0 END)                             AS stale_tables,
       max(last_modified)                                                 AS last_write_any_table
FROM inv
WHERE storage_class = 'root'
GROUP BY database
ORDER BY usd_per_month DESC NULLS LAST

## 5 · Most expensive tables

`action` is a suggestion: a stale table may still be read, so every DROP needs its owner's yes.
`saving_usd_per_month` is the whole table for a DROP and the old-version overhead for a VACUUM.

In [0]:
%sql
-- The 100 most expensive tables, with what to do about each
SELECT full_name,
       round(disk_bytes / power(1024, 3), 1)          AS disk_gib,
       usd(disk_bytes)                                 AS usd_per_month,
       round(live_bytes / power(1024, 3), 1)          AS live_gib,
       round(disk_bytes / nullif(live_bytes, 0), 1)   AS disk_to_live,
       CASE WHEN shallow_src                           THEN 'KEEP FILES: shallow-clone source, fix clones first'
            WHEN stale                                 THEN 'DROP candidate (confirm owner)'
            WHEN overhead_bytes > 0.5 * disk_bytes     THEN 'VACUUM'
            ELSE 'keep' END                            AS action,
       usd(CASE WHEN shallow_src THEN 0
                WHEN stale THEN disk_bytes
                ELSE overhead_bytes END)               AS saving_usd_per_month,
       last_modified, write_ops, last_vacuum_ts,
       coalesce(last_writer_job_name, last_writer_user) AS last_writer,
       table_type
FROM inv
WHERE storage_class = 'root'
ORDER BY disk_bytes DESC NULLS LAST
LIMIT 100

In [0]:
%sql
-- Same table name in several databases: usually copies of one source layer
SELECT table_name, count(*) AS copies,
       round(sum(disk_bytes) / power(1024, 4), 3) AS disk_tib,
       usd(sum(disk_bytes))                       AS usd_per_month,
       usd(sum(disk_bytes) - max(disk_bytes))     AS usd_if_one_copy_kept,
       concat_ws(', ', sort_array(collect_list(database))) AS in_databases
FROM inv
WHERE storage_class = 'root'
GROUP BY table_name
HAVING count(*) > 1
ORDER BY usd_per_month DESC NULLS LAST
LIMIT 30

## 6 · Orphan directories, log destinations, writers

In [0]:
if orph is not None:
    display(spark.sql("""
      SELECT path, round(bytes / power(1024, 3), 1) AS gib, usd(bytes) AS usd_per_month, files, parent_location, error
      FROM orphans ORDER BY bytes DESC NULLS LAST"""))
else:
    print("No orphan_dirs rows")

In [0]:
if logs is not None:
    display(spark.sql("""
      SELECT destination, round(max(destination_bytes) / power(1024, 3), 1) AS gib,
             usd(max(destination_bytes)) AS usd_per_month, count(*) AS clusters_and_jobs,
             concat_ws(', ', slice(sort_array(collect_set(owner_name)), 1, 8)) AS used_by
      FROM log_conf WHERE storage_class = 'root' AND destination IS NOT NULL
      GROUP BY destination ORDER BY max(destination_bytes) DESC NULLS LAST"""))
else:
    print("No cluster_log_conf rows")

In [0]:
if jobs is not None:
    display(spark.sql("""
      SELECT coalesce(i.last_writer_job_name, '(interactive: ' || i.last_writer_user || ')') AS writer,
             i.last_writer_job_id AS job_id, j.creator, j.schedule_cron, j.schedule_paused,
             j.last_run_start, j.last_run_state,
             count(*) AS tables, usd(sum(i.disk_bytes)) AS usd_per_month,
             usd(sum(i.overhead_bytes)) AS vacuumable_usd, max(i.last_write_ts) AS last_write,
             concat_ws(', ', slice(sort_array(collect_set(i.database)), 1, 8)) AS databases
      FROM inv i LEFT JOIN jobs j ON i.last_writer_job_id = j.job_id
      WHERE i.storage_class = 'root' AND i.last_write_ts IS NOT NULL
      GROUP BY ALL ORDER BY sum(i.disk_bytes) DESC NULLS LAST"""))

## 7 · Summary for the ticket

Copy the printed Markdown into the SEACO-6161 comment.

In [0]:
def top(sql, n=10):
    return spark.sql(sql).limit(n).collect()


top_tables = top("""SELECT full_name, disk_bytes, live_bytes, stale, shallow_src, overhead_bytes FROM inv
                    WHERE storage_class = 'root' ORDER BY disk_bytes DESC NULLS LAST""")
top_dbs = top("""SELECT database, sum(disk_bytes) b, count(*) n FROM inv WHERE storage_class = 'root'
                 GROUP BY database ORDER BY b DESC NULLS LAST""")
top_dirs = top("SELECT prefix, tib, usd_per_month, not_in_tables_tib FROM dir_cost "
               "WHERE level = 2 ORDER BY usd_per_month DESC") if scan_run else []

saving = stale_disk + active_over + orphan_b + log_b
md = [f"## Cost report: `{WORKSPACE_NAME}` (`{ROOT_STORAGE_ACCOUNT}`)", "",
      f"Built from audit runs: table inventory `{inv_run}` ({inv_ts:%Y-%m-%d}), "
      + (f"dbfs:/ scan `{scan_run}` ({scan_ts:%Y-%m-%d}, coverage "
         + ("unknown" if SCAN_COVERAGE is None else f"{SCAN_COVERAGE:.0%}") + ")." if scan_run else "no dbfs:/ scan."),
      f"Effective rate ${RATE * GIB:.5f}/GiB-month (${AZURE_MONTHLY_COST_USD:,.0f}/mo for {AZURE_REPORTED_TIB} TiB).", "",
      "### Where the money goes", "", "| | What | TiB | % | $/month | Action |", "|---|---|---:|---:|---:|---|"]
for k, n, b, a in sorted(BUCKET_ROWS, key=lambda r: -r[2]):
    md.append(f"| {k} | {n} | {b / TIB:,.2f} | {100 * b / AZURE_BYTES:.0f}% | ${usd(b):,.0f} | {a} |")
md += ["", f"**Reachable saving (A+B+E+F): {saving / TIB:,.2f} TiB ≈ ${usd(saving):,.0f}/month "
           f"(${12 * usd(saving):,.0f}/year)**, before G and H are looked at.", ""]
if top_dirs:
    md += ["### Most expensive directories (level 2)", "", "| Directory | TiB | $/month | not in tables, TiB |",
           "|---|---:|---:|---:|"]
    md += [f"| `{r.prefix}` | {r.tib:,.2f} | ${r.usd_per_month:,.0f} | {r.not_in_tables_tib:,.2f} |" for r in top_dirs]
    md.append("")
md += ["### Most expensive databases", "", "| Database | Tables | TiB on disk | $/month |", "|---|---:|---:|---:|"]
md += [f"| `{r.database}` | {r.n} | {(r.b or 0) / TIB:,.2f} | ${usd(r.b):,.0f} |" for r in top_dbs]
md += ["", "### Most expensive tables", "", "| Table | Disk GiB | Live GiB | $/month | Action |",
       "|---|---:|---:|---:|---|"]
for r in top_tables:
    act = ("keep files (shallow-clone source)" if r.shallow_src else "DROP candidate" if r.stale
           else "VACUUM" if (r.overhead_bytes or 0) > 0.5 * (r.disk_bytes or 0) else "keep")
    md.append(f"| `{r.full_name}` | {(r.disk_bytes or 0) / GIB:,.0f} | {(r.live_bytes or 0) / GIB:,.0f} "
              f"| ${usd(r.disk_bytes):,.0f} | {act} |")
REPORT_MD = "\n".join(md)

In [0]:
print(REPORT_MD)